# Chapter 9 — Trajectory Search

**Book alignment:** current Chapter 9 · internal demo `Stage 08`

The shared demo package calls this **Stage 08** internally. The notebook number follows the book chapter number; the internal stage number is one lower.

**Question this notebook isolates:** Can the runtime branch over isolated partial trajectories, allocate compute, prune, and still keep commitment outside the search tree?


## Three different inference regimes

```text
sequential scaling   one trajectory gets more work
leaf-level scaling   complete candidates are compared (Notebook 02)
prefix-level search  branch before completion and preserve partial futures
```


In [ ]:
from pathlib import Path
import sys

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'demo' / 'agents-from-first-principles').exists():
            return candidate
    raise RuntimeError('Run this notebook from a checkout containing demo/agents-from-first-principles')

REPO_ROOT = find_repo_root(Path.cwd().resolve())
DEMO_ROOT = REPO_ROOT / 'demo' / 'agents-from-first-principles'
sys.path.insert(0, str(DEMO_ROOT))

from first_principles_agent.runtime import RuntimeState
from first_principles_agent.search import (
    AllocationPolicy, FixedExpansionPolicy, IsolatedEnvironmentFactory,
    PruningPolicy, SearchController, SearchTransition,
)


## Controlled search tree

Branch `A` looks strong early but cannot reach the controlled target. Branch `B` looks mediocre early but can. This lets us distinguish generation failure from pruning failure.


In [ ]:
tree = FixedExpansionPolicy({
    'root': (
        SearchTransition('A', 'obvious parser edit', 0.9, success_reachable=False, workspace_updates=(('parser.py', 'candidate A'),)),
        SearchTransition('B', 'inspect delimiter assumptions', 0.4, success_reachable=True, workspace_updates=(('parser.py', 'candidate B'),)),
    ),
    'A': (SearchTransition('A1', 'delete failing test', 0.95, terminal=True, goal_satisfied=False, success_reachable=False),),
    'B': (SearchTransition('B1', 'patch delimiter parser', 0.8, terminal=True, goal_satisfied=True, success_reachable=True),),
})

root_state = RuntimeState(facts=frozenset({'repo_available'}))
factory = IsolatedEnvironmentFactory()

def run_search(width):
    return SearchController(expansion=tree, pruning=PruningPolicy(width), max_depth=2).run(
        root_state, factory, workspace={'parser.py': 'original'}
    )

narrow = run_search(1)
wide = run_search(2)


## Experiment 1 - pruning regret

The narrow search generates the promising route but kills it before it can be expanded. The wider search preserves it and reaches `B1`.


In [ ]:
assert 'B' in narrow.generated
assert 'B' in narrow.pruned
assert narrow.pruned_reachable == ('B',)
assert narrow.pruning_regret is True
assert narrow.selected is None

assert 'B' in wide.retained
assert 'B' in wide.expanded
assert wide.verified == ('B1',)
assert wide.selected == 'B1'
assert wide.pruning_regret is False

{
    'generated_narrow': narrow.generated,
    'retained_narrow': narrow.retained,
    'pruning_regret': narrow.pruning_regret,
    'verified_wide': wide.verified,
    'cost_per_controlled_success': wide.cost_per_verified_success,
}


## Experiment 2 - allocate compute where the decision is unresolved

A score near 0.5 is treated as more uncertain than one near an extreme. The allocation policy therefore gives branch `B` more expansion budget than high-confidence `A`.


In [ ]:
assert wide.allocations['B'] > wide.allocations['A']
wide.allocations


## Experiment 3 - sibling branches are isolated

Search comparisons are meaningless if one branch can mutate the state observed by another. Each child therefore receives an immutable forked snapshot.


In [ ]:
root = factory.root(root_state, {'parser.py': 'original'})
a = factory.fork(root, SearchTransition('A', 'branch A', 0.5, workspace_updates=(('parser.py', 'A edit'),)))
b = factory.fork(root, SearchTransition('B', 'branch B', 0.5, workspace_updates=(('parser.py', 'B edit'),)))

assert dict(root.snapshot.workspace)['parser.py'] == 'original'
assert dict(a.snapshot.workspace)['parser.py'] == 'A edit'
assert dict(b.snapshot.workspace)['parser.py'] == 'B edit'
assert a.fingerprint != b.fingerprint


## Search may branch freely; commitment is a later boundary

The root workspace in the isolation experiment remains `original` while children contain different previews. That is the book's **branch freely, commit once** rule in executable form: search may compare many futures, but selecting a branch does not itself mutate the real workspace.

Real mutation belongs after search, through the ordinary authorization/execution boundary, followed by fresh verification of the committed state.


## Experiment 4 - do not pay twice for the same effective state


In [ ]:
same_root = factory.root(RuntimeState(), {'parser.py': 'original'})
low = factory.fork(same_root, SearchTransition('low', 'same state', 0.2))
high = factory.fork(same_root, SearchTransition('high', 'same state', 0.8))
decision = PruningPolicy(2).prune((low, high), __import__('first_principles_agent.search', fromlist=['RecordedNodeEvaluator']).RecordedNodeEvaluator())
assert tuple(node.id for node in decision.retained) == ('high',)
assert tuple(node.id for node in decision.duplicates) == ('low',)


## What was earned

The agent can now branch before completion, preserve multiple continuable states, allocate compute selectively, deduplicate equivalent states, and expose pruning regret. Branches are isolated so comparisons remain meaningful.

The `TerminalOracle` used here is only a deterministic oracle for the controlled search experiment. It does **not** establish real-world success. Notebook 10 / Chapter 10 owns evidence-backed verification and integrity.
